In [ ]:
!pip install langchain youtube-transcript-api google-genai tiktoken  langchain-google-genai langchain-community faiss-cpu

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_18821/978820515.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

# Step 1a - Indexing (Document Ingestion)

In [11]:
video_id = "nXf63vN73BE"

api = YouTubeTranscriptApi()
transcript = api.fetch(video_id, languages=["en"])
transcript_list = transcript.to_raw_data()
transcript_text = " ".join(
    item["text"] for item in transcript_list
)

print(transcript_text)

- Hey, I'm John Kanell and
today on Preppy Kitchen I'm showing you how to make some quick and easy air
fryer potato wedges. So let's get started! First off grab two
beautiful russet potatoes and we're gonna scrub them underwater so they're nice and clean.
(water trickling) We want the skins on 'em though. Look at that! Before and after. Scrub-a-dub-dub, two potatoes in a tub. Scrubbing those potatoes
was so satisfying. They look 10 years younger. Alright, grab a sharp knife and we're gonna cut these potatoes lengthwise down the middle. (blade clacks) Now we're gonna quarter each of these halves for a total of eighths. You do wanna have a nice,
beautiful wedge though, so you know, cut nicely. All right, continue the process. This is the easiest recipe by the way. And if you haven't had like
an air fryer potato wedge, they're like french fries, but bigger and even more delicious. They're really nicely seasoned. It's a great easy side to do. And you know, we all have like a potato or two 

# Step 1b - Indexing (Text Splitting)

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript_text])

In [14]:
len(chunks)

8

In [15]:
chunks[0]

Document(metadata={}, page_content="- Hey, I'm John Kanell and\ntoday on Preppy Kitchen I'm showing you how to make some quick and easy air\nfryer potato wedges. So let's get started! First off grab two\nbeautiful russet potatoes and we're gonna scrub them underwater so they're nice and clean.\n(water trickling) We want the skins on 'em though. Look at that! Before and after. Scrub-a-dub-dub, two potatoes in a tub. Scrubbing those potatoes\nwas so satisfying. They look 10 years younger. Alright, grab a sharp knife and we're gonna cut these potatoes lengthwise down the middle. (blade clacks) Now we're gonna quarter each of these halves for a total of eighths. You do wanna have a nice,\nbeautiful wedge though, so you know, cut nicely. All right, continue the process. This is the easiest recipe by the way. And if you haven't had like")

# Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [16]:
vector_store = FAISS.from_documents(chunks,embeddings)

In [17]:
vector_store.index_to_docstore_id

{0: 'c0aea1bd-22bb-41fb-8970-845357b01691',
 1: '267c9b9e-3b64-40c9-88ea-3a5e7d837d48',
 2: '24b2eade-598a-4530-992a-664b64b42539',
 3: 'f2740a15-e7bf-48ab-8f80-602a829a7bd6',
 4: 'eeedc5bf-18b9-44e7-96c7-cbe02845427e',
 5: '863531e1-df0c-4eae-a0b7-629cdd4f54eb',
 6: 'e043a54b-32f3-4e91-809f-3c7b78ab60fe',
 7: '61c9c3d1-2433-4598-9ae6-e280794da04f'}

# Step 2 - Retrieval

In [20]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [22]:
retriever.invoke("What is the recipe")

[Document(id='c0aea1bd-22bb-41fb-8970-845357b01691', metadata={}, page_content="- Hey, I'm John Kanell and\ntoday on Preppy Kitchen I'm showing you how to make some quick and easy air\nfryer potato wedges. So let's get started! First off grab two\nbeautiful russet potatoes and we're gonna scrub them underwater so they're nice and clean.\n(water trickling) We want the skins on 'em though. Look at that! Before and after. Scrub-a-dub-dub, two potatoes in a tub. Scrubbing those potatoes\nwas so satisfying. They look 10 years younger. Alright, grab a sharp knife and we're gonna cut these potatoes lengthwise down the middle. (blade clacks) Now we're gonna quarter each of these halves for a total of eighths. You do wanna have a nice,\nbeautiful wedge though, so you know, cut nicely. All right, continue the process. This is the easiest recipe by the way. And if you haven't had like"),
 Document(id='24b2eade-598a-4530-992a-664b64b42539', metadata={}, page_content="kind of a gummy texture on you

# Step 3 - Augmentation

In [23]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=api_key
)

In [24]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [27]:
question = "what recipe is mentioned"
retrieved_docs    = retriever.invoke(question)
retrieved_docs

[Document(id='eeedc5bf-18b9-44e7-96c7-cbe02845427e', metadata={}, page_content="all the residual water off. I want these to be crispy,\ngolden and delicious. Not soggy and sad. That'd be sad. And don't think I'm\ngonna pop these bone dry potato wedges into a wet bowl. This will get dried out as well. Hmm. (cheerful music continues) Alright, my bowl is dry\nand so are the potatoes. So transfer them over. (potatoes plop) And now we are gonna drizzle them with a tablespoon of olive oil. If you feel like you can\nnever get even coverage by drizzling, you can use a mister. I have a really great micro\nmister that somebody broke, but otherwise I'd be doing this (imitates spraying) and\nit's a beautiful spritz. Okay. Okay. Tossed and nicely coated. Now it's time to give\nthese potatoes some flavor. Starting with one teaspoon of salt, gently sprinkle all over\nand then give 'em a mix and continue sprinkling. Don't just dump the salt"),
 Document(id='61c9c3d1-2433-4598-9ae6-e280794da04f', metad

In [28]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"all the residual water off. I want these to be crispy,\ngolden and delicious. Not soggy and sad. That'd be sad. And don't think I'm\ngonna pop these bone dry potato wedges into a wet bowl. This will get dried out as well. Hmm. (cheerful music continues) Alright, my bowl is dry\nand so are the potatoes. So transfer them over. (potatoes plop) And now we are gonna drizzle them with a tablespoon of olive oil. If you feel like you can\nnever get even coverage by drizzling, you can use a mister. I have a really great micro\nmister that somebody broke, but otherwise I'd be doing this (imitates spraying) and\nit's a beautiful spritz. Okay. Okay. Tossed and nicely coated. Now it's time to give\nthese potatoes some flavor. Starting with one teaspoon of salt, gently sprinkle all over\nand then give 'em a mix and continue sprinkling. Don't just dump the salt\n\ntransport to a serving dish, garnish with some chopped parsley and freshly grated Parmesan,\nand they're ready to enjoy. (cheerful music 

In [29]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [30]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      all the residual water off. I want these to be crispy,\ngolden and delicious. Not soggy and sad. That'd be sad. And don't think I'm\ngonna pop these bone dry potato wedges into a wet bowl. This will get dried out as well. Hmm. (cheerful music continues) Alright, my bowl is dry\nand so are the potatoes. So transfer them over. (potatoes plop) And now we are gonna drizzle them with a tablespoon of olive oil. If you feel like you can\nnever get even coverage by drizzling, you can use a mister. I have a really great micro\nmister that somebody broke, but otherwise I'd be doing this (imitates spraying) and\nit's a beautiful spritz. Okay. Okay. Tossed and nicely coated. Now it's time to give\nthese potatoes some flavor. Starting with one teaspoon of salt, gently sprinkle all over\nand then give 'em a mix 

# Step 4 - Generation

In [31]:
answer = llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': 'Based on the provided context, the recipe mentioned is for **air fryer potato wedges**.', 'extras': {'signature': 'EukNCuYNAWkUfROJ0EUJgLDzQbY+BZiSGs016vsaMsPqbXESnA0f+8n96alq44f+HzuiuHE8iW6zabR8luOYVmCw1Q7+tJOh2iqXdcF6mWx6Gkh5oNzO2LezXt3NOcJfJ1ImMhJD+lCv5xJl9BXijCrhBI7HXIPxEp/HEGBUcbeL2uuQRUbN1hh81+lamVR1HIY5ngCyPgTwrOs0J9C1l7hwNhTKxE8wrmI+81LdKM1f2iQhH+9pk7DHLja1xhLCeriiB1ePSUytWp0spqrj+I/yopy3qICGUzI5s4mI1se4dZ5P1EcL6U1xmiYXZfrdPlGEP6512pRPE9m6h/7QThc0CuEVDiIr2XoT9Zy5iiRaVIje8zvDwHwtrdTe8BWsHslHeaYLIzJWAFMAMCmJALgPj70HTc/KH4Md83tT1hFTYu3a0zfMTXPmnGyoGNPIK/MyNJOIr8meDQ2lkLb/2iZZgOsnqw8+425vnODLfZwPVvn93ybsJb21voZoJ77EA0ml4o1IhoifKIKWIYv7BTJsT9jcm581TGsgp0ejzzt8/zb+ht7J0Yy6wBF6G6Mx5n2Iolf+YxrBoPSE3JK0u2P5z5gjCoKCP1fjQ2RdDldfAC+deilSjm73vzQ4lPooa9ej1XJGTfbAIUlZD8VDnccKDxW3lWxF6nliaZH55zjPLpMkPb3BFrECMQMuQFXx8cuKh04623B/mQpiX5ITyehtDgg0Slaf2sXZJEULnUBR14Ot7DqRNfRnaUOemhqnB1vubjKMEe9wIOXri4m3TolT3fYUl2PZ3t8aF+kg2HeonzDWFrnOdV7D6h8v7dmQ56B9VE6MMJM2pZwI7QGT2eZWS7

# Building a Chain

In [32]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [33]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [34]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [35]:
parallel_chain.invoke('How many potions are mentioned')

{'context': "all the residual water off. I want these to be crispy,\ngolden and delicious. Not soggy and sad. That'd be sad. And don't think I'm\ngonna pop these bone dry potato wedges into a wet bowl. This will get dried out as well. Hmm. (cheerful music continues) Alright, my bowl is dry\nand so are the potatoes. So transfer them over. (potatoes plop) And now we are gonna drizzle them with a tablespoon of olive oil. If you feel like you can\nnever get even coverage by drizzling, you can use a mister. I have a really great micro\nmister that somebody broke, but otherwise I'd be doing this (imitates spraying) and\nit's a beautiful spritz. Okay. Okay. Tossed and nicely coated. Now it's time to give\nthese potatoes some flavor. Starting with one teaspoon of salt, gently sprinkle all over\nand then give 'em a mix and continue sprinkling. Don't just dump the salt\n\nthese potatoes some flavor. Starting with one teaspoon of salt, gently sprinkle all over\nand then give 'em a mix and continu

In [36]:
parser = StrOutputParser()

In [37]:
main_chain = parallel_chain | prompt | llm | parser

In [38]:
main_chain.invoke('Can you summarize the video')

"Based on the provided transcript, here is a summary of the video:\n\n* **Preparation:** John Kanell scrubs two russet potatoes (leaving the skins on) and cuts them lengthwise into eighths to create wedges. \n* **Ice Bath:** He soaks the wedges in a bowl of ice water for 10 minutes. This draws out the starch to prevent a gummy texture, stop discoloration, and ensure the potatoes get crisp.\n* **Rinsing & Drying:** Using tongs, he removes the potatoes, dumps the cloudy starch water, gives the wedges a final rinse in fresh water, and thoroughly dries both the wedges and the bowl so the potatoes don't end up soggy.\n* **Oil & Seasoning:** He transfers the dry wedges back to the dry bowl, drizzling them with a tablespoon of olive oil to coat them evenly, and starts seasoning them with a teaspoon of salt. \n\n*(Note: The transcript cuts off mid-recipe while he is applying the seasoning.)*"